<a href="https://colab.research.google.com/github/javersifaz05-rgb/PRACT-HOME/blob/main/plant_diseases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Path to dataset files: /kaggle/input/new-plant-diseases-dataset


In [ ]:
import os

train_path = path + "/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
print(os.path.exists(train_path))

True


In [ ]:
file = open("plant_disease_data.csv", "w")
file.write("ID,Plant,Disease,Image_Count\n")

number = 1
for class_name in sorted(os.listdir(train_path)):
    parts = class_name.split("___")
    plant = parts[0].replace("_", " ").replace(",", "")
    disease = parts[1].replace("_", " ")
    count = len(os.listdir(train_path + "/" + class_name))
    file.write("D" + str(number) + "," + plant + "," + disease + "," + str(count) + "\n")
    number += 1

file.close()
print("CSV  are ready :", number - 1, " pic record")

CSV  are ready : 38  pic record


In [8]:
import numpy as np

# Tuple: Fixed severity or count categories as required in rubric
COUNT_CATEGORIES = ("Low Data", "Medium Data", "High Data")

class Plant:
    def __init__(self, plant_id, name, disease, count):
        self.plant_id = plant_id
        self.name = name
        self.disease = disease
        self.count = count

    def get_status(self):
        if self.count < 1500:
            return COUNT_CATEGORIES[0]
        elif self.count <= 2000:
            return COUNT_CATEGORIES[1]
        else:
            return COUNT_CATEGORIES[2]

    def show(self):
        status = self.get_status()
        print(self.plant_id, "|", self.name, "|", self.disease, "|", self.count, "|", status)

print("Step 1 Complete: NumPy imported and Plant class created.")
print("Fixed Categories Tuple:", COUNT_CATEGORIES)

Step 1 Complete: NumPy imported and Plant class created.
Fixed Categories Tuple: ('Low Data', 'Medium Data', 'High Data')


In [11]:
import os
import kagglehub

# Step A: If CSV does not exist, create it first
if not os.path.exists("plant_disease_data.csv"):
    path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
    train_path = path + "/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"

    file = open("plant_disease_data.csv", "w")
    file.write("ID,Plant,Disease,Image_Count\n")

    number = 1
    for class_name in sorted(os.listdir(train_path)):
        parts = class_name.split("___")
        plant = parts[0].replace("_", " ").replace(",", "")
        disease = parts[1].replace("_", " ")
        count = len(os.listdir(train_path + "/" + class_name))
        file.write("D" + str(number) + "," + plant + "," + disease + "," + str(count) + "\n")
        number += 1
    file.close()
    print("CSV created successfully with", number - 1, "records.")

# Step B: Load data from CSV into List and Set
plants = []
ids = set()

def load_data():
    try:
        file = open("plant_disease_data.csv", "r")
        lines = file.readlines()
        file.close()
    except FileNotFoundError:
        print("Error: File not found!")
        return

    for line in lines[1:]:
        parts = line.strip().split(",")
        if len(parts) != 4:
            continue

        try:
            count = int(parts[3])
        except ValueError:
            continue

        if parts[0] in ids:
            continue

        obj = Plant(parts[0], parts[1], parts[2], count)
        plants.append(obj)
        ids.add(parts[0])

    print("Total records loaded into List:", len(plants))
    print("Total unique IDs in Set       :", len(ids))

load_data()

100%|██████████| 2.70G/2.70G [00:29<00:00, 97.5MB/s]


Extracting files...
CSV created successfully with 38 records.
Total records loaded into List: 38
Total unique IDs in Set       : 38


In [12]:
plant_summary_dict = {}  # Dictionary: {Plant_Name: Total_Images}
record_tuples = []  # List of Tuples: (ID, Plant, Disease)

for p in plants:
    # Dictionary calculation using if/else
    if p.name in plant_summary_dict:
        plant_summary_dict[p.name] = plant_summary_dict[p.name] + p.count
    else:
        plant_summary_dict[p.name] = p.count

    # Creating Tuple for each record
    record_tuples.append((p.plant_id, p.name, p.disease))

print("--- Dictionary Output (Plant : Total Images) ---")
for plant_name, total_images in plant_summary_dict.items():
    print(plant_name, ":", total_images)

print("\n--- Sample Tuple Records ---")
for t in record_tuples[:3]:
    print(t)

--- Dictionary Output (Plant : Total Images) ---
Apple : 7771
Blueberry : 1816
Cherry (including sour) : 3509
Corn (maize) : 7316
Grape : 7222
Orange : 2010
Peach : 3566
Pepper bell : 3901
Potato : 5702
Raspberry : 1781
Soybean : 2022
Squash : 1736
Strawberry : 3598
Tomato : 18345

--- Sample Tuple Records ---
('D1', 'Apple', 'Apple scab')
('D2', 'Apple', 'Black rot')
('D3', 'Apple', 'Cedar apple rust')


In [13]:
# NumPy Analysis of Image Counts
counts_list = []
for p in plants:
    counts_list.append(p.count)

# Convert list to NumPy array
counts_array = np.array(counts_list)

print("--- NumPy Statistical Summary ---")
print("Total Classes      :", len(counts_array))
print("Total Images (Sum) :", np.sum(counts_array))
print("Average Images     :", round(np.mean(counts_array), 2))
print("Maximum Images     :", np.max(counts_array))
print("Minimum Images     :", np.min(counts_array))
print("Median Value       :", round(np.median(counts_array), 2))
print("Standard Deviation :", round(np.std(counts_array), 2))

--- NumPy Statistical Summary ---
Total Classes      : 38
Total Images (Sum) : 70295
Average Images     : 1849.87
Maximum Images     : 2022
Minimum Images     : 1642
Median Value       : 1844.5
Standard Deviation : 104.32


In [14]:
# 1. Search Function using loop and if/else
def search_by_plant(plant_name):
    print("--- Search Results for:", plant_name, "---")
    found = False
    for p in plants:
        if p.name.lower() == plant_name.lower():
            p.show()
            found = True
    if not found:
        print("No record found for", plant_name)

# 2. Health Status Check Function
def check_health_status():
    healthy = 0
    diseased = 0
    for p in plants:
        if "healthy" in p.disease.lower():
            healthy += 1
        else:
            diseased += 1
    print("\n--- Health Status Breakdown ---")
    print("Healthy Categories  :", healthy)
    print("Diseased Categories :", diseased)

# Testing the functions
search_by_plant("Tomato")
check_health_status()

--- Search Results for: Tomato ---
D29 | Tomato | Bacterial spot | 1702 | Medium Data
D30 | Tomato | Early blight | 1920 | Medium Data
D31 | Tomato | Late blight | 1851 | Medium Data
D32 | Tomato | Leaf Mold | 1882 | Medium Data
D33 | Tomato | Septoria leaf spot | 1745 | Medium Data
D34 | Tomato | Spider mites Two-spotted spider mite | 1741 | Medium Data
D35 | Tomato | Target Spot | 1827 | Medium Data
D36 | Tomato | Tomato Yellow Leaf Curl Virus | 1961 | Medium Data
D37 | Tomato | Tomato mosaic virus | 1790 | Medium Data
D38 | Tomato | healthy | 1926 | Medium Data

--- Health Status Breakdown ---
Healthy Categories  : 12
Diseased Categories : 26


In [15]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

#destinition
source_file = "plant_disease_data.csv"
#drive folder
drive_destination = (
    "/content/drive/MyDrive/python mid P/plant_disease_data.csv"
)

if os.path.exists(source_file):
    shutil.copy(source_file, drive_destination)
    print("Success! File uploaded and saved to your 'python mid P' folder at:")
    print(drive_destination)
else:
    print("Error: plant_disease_data.csv file was not found in the current session.")

Mounted at /content/drive
Success! File uploaded and saved to your 'python mid P' folder at:
/content/drive/MyDrive/python mid P/plant_disease_data.csv
